# General Overview

NASA DONKI (Database Of Notifications, Knowledge, Information) Space Weather Events is a tabular dataset of space weather activity collected by NASA's Community Coordinated Modeling Center. It includes events such as coronal mass ejections, geomagnetic storms, interplanetary shocks, high-speed streams, and solar energetic particle events.

The dataset is useful for studying the full Sun-to-Earth chain of space weather because it preserves causal relationships through the `linked_events` field. That makes it possible to trace how one event may trigger another, for example CME -> IPS -> GST.

A few useful points about the dataset:
- coverage starts in 2010 and continues to the present
- the current Hugging Face snapshot contains 10,903 events
- it is suitable for tabular classification and time-series forecasting tasks
- it is updated daily, so the row count can change over time

## How This Notebook Fits the Project

**Project question:** *Predict whether a geomagnetic storm will occur on Earth from solar activity* — a binary classification problem.

DONKI is the dataset that ties the chain together, so this EDA is framed around that target. The natural unit of prediction here is a **CME**: most geomagnetic storms (GSTs) are driven by a CME that left the Sun days earlier, and the `linked_events` field tells us *which* CMEs were later linked to a storm. That gives us a ready-made binary label per CME — `led_to_storm` (True/False) — built at the end of this notebook. The EDA below works toward three things a classifier needs:

1. **Candidate features** — which CME properties (speed, width, source direction) look predictive of a storm.
2. **Label quality and class balance** — how many CMEs actually lead to storms, and how imbalanced that target is.
3. **Data hazards** — structural vs. genuine missing values, outliers, and possible leakage — so feature engineering starts from clean assumptions.

## Terms Used in This Notebook

A quick reference for the terms that appear in the columns and plots below:

- CME: coronal mass ejection, a burst of plasma and magnetic field from the Sun (the `event_type` we predict from)
- GST: geomagnetic storm, a disturbance in Earth's magnetic field — the outcome we want to predict
- IPS / HSS / SEP: interplanetary shock / high-speed stream / solar energetic particle event — the other event classes in `event_type`
- Kp index: a global scale that measures geomagnetic disturbance, recorded in `gst_max_kp`
- active region: a magnetically active area on the Sun where flares and CMEs begin (`active_region`)
- heliographic coordinates: a Sun-based coordinate system describing where a CME originates (`source_location`, `cme_latitude`, `cme_longitude`)
- geoeffective: likely to produce effects on Earth — a CME near central longitude (Earth-facing) is more geoeffective

In [ ]:
%pip install -q -r requirements.txt

In [93]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

In [ ]:
%pip install -q datasets
from datasets import load_dataset

ds = load_dataset("juliensimon/donki-space-weather-events", split="train")
df = ds.to_pandas()

In [95]:
print(f"{df.shape[0]} rows, {df.shape[1]} columns") 

10903 rows, 17 columns


In [97]:
df.head(7)

,event_type,activity_id,start_time,source_location,active_region,note,link,cme_speed_kms,cme_half_angle_deg,cme_latitude,cme_longitude,cme_type,cme_time_21_5,cme_measurement,linked_events,gst_max_kp,gst_kp_count
0,IPS,2010-01-20T20:20:00-IPS-001,2010-01-20 20:20:00+00:00,NaN,NaN,STEREO B,https://kauai.ccmc.gsfc.nasa.gov/DONKI/view/IP...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
1,IPS,2010-02-05T03:33:00-IPS-001,2010-02-05 03:33:00+00:00,NaN,NaN,STEREO A,https://kauai.ccmc.gsfc.nasa.gov/DONKI/view/IP...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
2,IPS,2010-02-14T07:50:00-IPS-001,2010-02-14 07:50:00+00:00,NaN,NaN,STEREO B,https://kauai.ccmc.gsfc.nasa.gov/DONKI/view/IP...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
3,IPS,2010-03-05T18:25:00-IPS-001,2010-03-05 18:25:00+00:00,NaN,NaN,STEREO A,https://kauai.ccmc.gsfc.nasa.gov/DONKI/view/IP...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN
4,CME,2010-04-03T09:54:00-CME-001,2010-04-03 09:54:00+00:00,S20E05,NaN,SDO images were unavailable. CME source locati...,https://kauai.ccmc.gsfc.nasa.gov/DONKI/view/CM...,620.0,26.0,7.0,8.0,C,2010-04-03 17:16:00+00:00,null,"2010-04-03T09:04:00-FLR-001, 2010-04-05T07:54:...",NaN,NaN
5,IPS,2010-04-05T07:54:00-IPS-001,2010-04-05 07:54:00+00:00,NaN,NaN,Earth,https://kauai.ccmc.gsfc.nasa.gov/DONKI/view/IP...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,2010-04-03T09:54:00-CME-001,NaN,NaN
6,GST,2010-04-05T12:00:00-GST-001,2010-04-05 12:00:00+00:00,NaN,NaN,NaN,https://kauai.ccmc.gsfc.nasa.gov/DONKI/view/GS...,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,7.0,1.0


In [ ]:
summary_rows = [
    {'column': 'event_type', 'dtype': str(df['event_type'].dtype), 'variable_type': 'categorical', 'represents': 'Event class (CME, GST, IPS, HSS, SEP)', 'missing_count': int(df['event_type'].isna().sum()), 'missing_pct': round(df['event_type'].isna().mean() * 100, 1)},
    {'column': 'activity_id', 'dtype': str(df['activity_id'].dtype), 'variable_type': 'categorical/id', 'represents': 'Unique DONKI event identifier', 'missing_count': int(df['activity_id'].isna().sum()), 'missing_pct': round(df['activity_id'].isna().mean() * 100, 1)},
    {'column': 'start_time', 'dtype': str(df['start_time'].dtype), 'variable_type': 'datetime', 'represents': 'Event start time in UTC', 'missing_count': int(df['start_time'].isna().sum()), 'missing_pct': round(df['start_time'].isna().mean() * 100, 1)},
    {'column': 'source_location', 'dtype': str(df['source_location'].dtype), 'variable_type': 'categorical/text', 'represents': 'Heliographic CME source location', 'missing_count': int(df['source_location'].isna().sum()), 'missing_pct': round(df['source_location'].isna().mean() * 100, 1)},
    {'column': 'active_region', 'dtype': str(df['active_region'].dtype), 'variable_type': 'categorical/id', 'represents': 'NOAA active region number', 'missing_count': int(df['active_region'].isna().sum()), 'missing_pct': round(df['active_region'].isna().mean() * 100, 1)},
    {'column': 'note', 'dtype': str(df['note'].dtype), 'variable_type': 'categorical/text', 'represents': 'Analyst note or observatory hint', 'missing_count': int(df['note'].isna().sum()), 'missing_pct': round(df['note'].isna().mean() * 100, 1)},
    {'column': 'link', 'dtype': str(df['link'].dtype), 'variable_type': 'categorical/url', 'represents': 'DONKI web page for the event', 'missing_count': int(df['link'].isna().sum()), 'missing_pct': round(df['link'].isna().mean() * 100, 1)},
    {'column': 'cme_speed_kms', 'dtype': str(df['cme_speed_kms'].dtype), 'variable_type': 'numeric', 'represents': 'CME speed in km/s', 'missing_count': int(df['cme_speed_kms'].isna().sum()), 'missing_pct': round(df['cme_speed_kms'].isna().mean() * 100, 1)},
    {'column': 'cme_half_angle_deg', 'dtype': str(df['cme_half_angle_deg'].dtype), 'variable_type': 'numeric', 'represents': 'CME angular half-width in degrees', 'missing_count': int(df['cme_half_angle_deg'].isna().sum()), 'missing_pct': round(df['cme_half_angle_deg'].isna().mean() * 100, 1)},
    {'column': 'cme_latitude', 'dtype': str(df['cme_latitude'].dtype), 'variable_type': 'numeric', 'represents': 'CME source latitude in degrees', 'missing_count': int(df['cme_latitude'].isna().sum()), 'missing_pct': round(df['cme_latitude'].isna().mean() * 100, 1)},
    {'column': 'cme_longitude', 'dtype': str(df['cme_longitude'].dtype), 'variable_type': 'numeric', 'represents': 'CME source longitude in degrees', 'missing_count': int(df['cme_longitude'].isna().sum()), 'missing_pct': round(df['cme_longitude'].isna().mean() * 100, 1)},
    {'column': 'cme_type', 'dtype': str(df['cme_type'].dtype), 'variable_type': 'categorical', 'represents': 'CME morphology class (S, C, O, R, ER)', 'missing_count': int(df['cme_type'].isna().sum()), 'missing_pct': round(df['cme_type'].isna().mean() * 100, 1)},
    {'column': 'cme_time_21_5', 'dtype': str(df['cme_time_21_5'].dtype), 'variable_type': 'datetime', 'represents': 'Estimated time CME reaches 21.5 solar radii', 'missing_count': int(df['cme_time_21_5'].isna().sum()), 'missing_pct': round(df['cme_time_21_5'].isna().mean() * 100, 1)},
    {'column': 'cme_measurement', 'dtype': str(df['cme_measurement'].dtype), 'variable_type': 'categorical/text', 'represents': 'CME measurement technique or note', 'missing_count': int(df['cme_measurement'].isna().sum()), 'missing_pct': round(df['cme_measurement'].isna().mean() * 100, 1)},
    {'column': 'linked_events', 'dtype': str(df['linked_events'].dtype), 'variable_type': 'categorical/text', 'represents': 'Comma-separated linked event IDs', 'missing_count': int(df['linked_events'].isna().sum()), 'missing_pct': round(df['linked_events'].isna().mean() * 100, 1)},
    {'column': 'gst_max_kp', 'dtype': str(df['gst_max_kp'].dtype), 'variable_type': 'numeric', 'represents': 'Maximum geomagnetic Kp index during GST', 'missing_count': int(df['gst_max_kp'].isna().sum()), 'missing_pct': round(df['gst_max_kp'].isna().mean() * 100, 1)},
    {'column': 'gst_kp_count', 'dtype': str(df['gst_kp_count'].dtype), 'variable_type': 'numeric', 'represents': 'Number of Kp readings during GST', 'missing_count': int(df['gst_kp_count'].isna().sum()), 'missing_pct': round(df['gst_kp_count'].isna().mean() * 100, 1)},
]

column_summary = pd.DataFrame(summary_rows)
column_summary

## Missing Values: Structural vs. Genuine

The column summary above counts nulls per column overall. To decide what to *do* about them, it's more useful to look at the **fraction missing per event type**. A null can mean two very different things:

- **Structurally not-applicable** — e.g. `cme_speed_kms` is null for a GST because a geomagnetic storm has no CME speed. The value isn't missing, it's *undefined*. These nulls are correct and should **not** be imputed or filled with 0 (a `cme_speed_kms` of 0 would falsely claim "a CME at rest").
- **Genuinely missing** — a column that is null *even within the event type it describes* (e.g. CME rows that lack `cme_longitude`). These are real candidates for imputation, flagging, or dropping.

The table below reads column-by-column: look for high values in the "wrong" event-type rows (structural) versus high values in the row the column belongs to (genuine missingness).

In [ ]:
null_fraction_by_type = (
    df.groupby("event_type")
      .apply(lambda g: g.isnull().mean(), include_groups=False)
      .round(2)
)
null_fraction_by_type

## Visualizing the Data

A few plots to get a feel for the dataset before modeling. We look at four things, in order:

1. **How common each event type is** — the dataset is dominated by CMEs, which is why CMEs are our unit of prediction.
2. **CME speed distribution** — speed is the headline CME feature and our strongest candidate storm predictor.
3. **Events over time** — activity should track the ~11-year solar cycle, which matters for how we split train/test data by time.
4. **Storm severity (peak Kp)** — the underlying intensity behind the binary "storm vs. no storm" label.

A note on filtering: every plot below first restricts to the relevant `event_type`, so we never plot the structural nulls discussed above (e.g. CME speed only exists for CME rows).

In [ ]:
event_counts = df["event_type"].value_counts()

fig, ax = plt.subplots(figsize=(7, 4))
event_counts.plot.bar(ax=ax, color="steelblue")
ax.bar_label(ax.containers[0])
ax.set_title("Number of Events by Type (2010–present)")
ax.set_xlabel("Event type")
ax.set_ylabel("Count")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
cmes = df[df["event_type"] == "CME"]

fig, ax = plt.subplots(figsize=(7, 4))
cmes["cme_speed_kms"].plot.hist(bins=50, ax=ax, color="steelblue")
ax.set_title("DONKI CME Speed Distribution")
ax.set_xlabel("CME speed (km/s)")
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()

In [ ]:
df["year"] = df["start_time"].dt.year
events_by_year = df.groupby(["year", "event_type"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(10, 5))
events_by_year.plot(ax=ax, marker="o", ms=3)
ax.set_title("DONKI Events by Type and Year")
ax.set_xlabel("Year")
ax.set_ylabel("Number of events")
ax.legend(title="Event type")
plt.tight_layout()
plt.show()

In [ ]:
storms = df[df["event_type"] == "GST"]

fig, ax = plt.subplots(figsize=(7, 4))
storms["gst_max_kp"].plot.hist(bins=range(0, 11), rwidth=0.9, ax=ax, color="indianred")
ax.axvline(5, color="black", linestyle="--", label="G1 storm threshold (Kp = 5)")
ax.set_title(f"Geomagnetic Storm Severity  (n = {len(storms)} storms)")
ax.set_xlabel("Peak Kp index")
ax.set_ylabel("Number of storms")
ax.set_xticks(range(0, 11))
ax.legend()
plt.tight_layout()
plt.show()

## Outlier Detection & Relationships

The plots above describe single columns. This section gives the group the tools to (a) **spot outliers** in the CME measurements and (b) **see relationships** between features and storm outcomes — the two things needed to make informed cleaning and modeling decisions.

All cells operate on a `cme` slice (CME events only) so we're working with genuinely-populated numeric columns, not structural nulls.

> An "outlier" here is a flag for review, not automatically an error. A 3,500 km/s CME is a real, extreme event — not noise to delete. Use these views to *decide* what each extreme value is, not to blindly remove it.

In [ ]:
cme = df[df["event_type"] == "CME"].copy()
num_cols = ["cme_speed_kms", "cme_half_angle_deg", "cme_latitude", "cme_longitude"]

cme[num_cols].describe().round(1)

In [ ]:
fig, axes = plt.subplots(1, len(num_cols), figsize=(15, 4))
for ax, col in zip(axes, num_cols):
    cme.boxplot(column=col, ax=ax)
    ax.set_title(col)
fig.suptitle("CME Feature Boxplots (outliers = points past the whiskers)")
plt.tight_layout()
plt.show()

In [ ]:
def iqr_outliers(s):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    mask = (s < lo) | (s > hi)
    return pd.Series({
        "n_outliers": int(mask.sum()),
        "pct": round(100 * mask.mean(), 1),
        "low_cut": round(lo, 1),
        "high_cut": round(hi, 1),
    })

cme[num_cols].apply(iqr_outliers).T

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.ravel(), num_cols):
    sns.histplot(cme[col].dropna(), kde=True, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cme[num_cols].corr(), annot=True, cmap="coolwarm",
            vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Correlation between CME features")
plt.tight_layout()
plt.show()

In [ ]:
order = [t for t in ["S", "C", "O", "R", "ER"] if t in cme["cme_type"].dropna().unique()]
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=cme, x="cme_type", y="cme_speed_kms", order=order, ax=ax)
ax.set_title("CME Speed by Type Class")
ax.set_xlabel("CME type (slow → extremely rapid)")
ax.set_ylabel("Speed (km/s)")
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
sns.scatterplot(data=cme, x="cme_longitude", y="cme_latitude", alpha=0.25, s=20, ax=ax)
ax.axvline(0, color="grey", lw=0.8)
ax.axhline(0, color="grey", lw=0.8)
ax.set_title("CME Source Direction (heliographic)")
ax.set_xlabel("Longitude (deg)  —  0 = Earth-facing")
ax.set_ylabel("Latitude (deg)")
plt.tight_layout()
plt.show()

## Building the Binary Target

This is the cell the whole project hinges on. We turn DONKI's causal links into the **binary classification label**: for each CME, did a geomagnetic storm later get linked to it?

The label is built without leakage from the future *of that CME* — it uses only the `linked_events` of GST rows to mark which CME `activity_id`s were storm-driving. The features a model would actually use (speed, width, source direction) are all measured at the CME's own start time, before any storm occurs.

Two things to read off the cell below, both critical for modeling:

- **Signal:** is there a real gap in CME speed between the "led to a storm" and "no storm" groups? If yes, speed is a usable feature.
- **Class balance:** what fraction of CMEs lead to storms? Expect heavy imbalance (storms are rare), which means accuracy is a misleading metric — we'll need precision/recall, and likely class weighting or resampling.

In [ ]:
gst_links = df.loc[df["event_type"] == "GST", "linked_events"].dropna()
linked_cme_ids = {
    tok.strip()
    for s in gst_links
    for tok in s.split(",")
    if "CME" in tok
}
cme["led_to_storm"] = cme["activity_id"].isin(linked_cme_ids)
cme["storm_outcome"] = cme["led_to_storm"].map({True: "Linked to storm", False: "No linked storm"})

fig, ax = plt.subplots(figsize=(7, 5))
sns.boxplot(data=cme, x="storm_outcome", y="cme_speed_kms",
            order=["No linked storm", "Linked to storm"], ax=ax)
ax.set_title("CME Speed vs. Whether a Geomagnetic Storm Followed")
ax.set_xlabel("")
ax.set_ylabel("CME speed (km/s)")
plt.tight_layout()
plt.show()

n = len(cme)
n_pos = int(cme["led_to_storm"].sum())
print(f"Binary target `led_to_storm` over {n} CMEs:")
print(f"  storm-linked (positive): {n_pos:>5}  ({100 * n_pos / n:.1f}%)")
print(f"  no storm     (negative): {n - n_pos:>5}  ({100 * (n - n_pos) / n:.1f}%)")
print(f"  imbalance ratio        : 1 : {round((n - n_pos) / max(n_pos, 1))}  (negative : positive)\n")

print("Median CME speed (km/s):")
print(cme.groupby("storm_outcome")["cme_speed_kms"].median().round(0))

## EDA Summary

**What the dataset is for:** DONKI tracks space-weather events and the causal links between them, so it lets us build a per-CME binary label — "did a geomagnetic storm follow?" — directly from `linked_events`.

**Findings**
- **What it represents:** a timeline of CMEs, IPS, HSS, SEP events, and GSTs from 2010 onward; CMEs dominate (~8,100 of ~10,900 rows), which is why they're our unit of prediction.
- **Missingness:** most nulls are *structural*, not errors. GST fields fill only for GST rows, CME fields only for CME rows, and `linked_events` is populated only when an event has causal links. These should not be imputed or zero-filled.
- **Outliers:** CME speed has the clearest tail; the IQR rule flags 376 CME speeds above ~1,080 km/s. These are real extreme events, not noise — keep them, they are likely the storm drivers.
- **Signal:** storm-linked CMEs are clearly faster than non-storm CMEs, so `cme_speed_kms` is a strong candidate feature; source longitude (Earth-facing) and half-angle are worth testing too.

**Implications for the binary-classification model**
- **Target:** `led_to_storm` per CME, built in the last cell from GST `linked_events`.
- **Class imbalance:** storm-linked CMEs are the minority class — report precision/recall/F1, not accuracy, and consider class weighting or resampling.
- **Time-aware split:** activity tracks the solar cycle, so split train/test by time rather than randomly to avoid leaking future activity.
- **Candidate features:** `cme_speed_kms`, `cme_half_angle_deg`, `cme_longitude`/`cme_latitude`, `cme_type`. Avoid using any GST-side field (e.g. `gst_max_kp`) as a feature — those are only known *after* a storm and would leak the label.
- **Next steps:** join the other datasets (solar-flare-events, solar-wind, space-weather-indices) on time to add upstream features, then start with logistic regression as a baseline before tree-based models.